In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =====================================================================================
# ENRICHMENT ANALYSIS FOR REVIEWER #2, CONCERN #12  (Colab, plain Python)
#   * Gene universe: genes measured on both platforms (GSE98793 and GSE63878)
#   * Libraries: Enrichr GO 2025 (BP/MF/CC), KEGG 2021, WikiPathways 2024, Reactome 2024
#   * Method: one-sided hypergeometric test, Benjamini-Hochberg FDR within each library
#   * Reports: overlap, term size, p, FDR, enrichment ratio, odds ratio, contributing genes
#   * Two gene lists analysed separately: 19 shared DEGs (primary) and 10 hub genes
# =====================================================================================
import os, re, io, datetime, numpy as np, pandas as pd, requests, matplotlib.pyplot as plt
from scipy.stats import hypergeom
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/research/p5-stress_depression/p5_stress_depression_journal-20260927T103230Z-1-001/p5_stress_depression_journal/"
MATRICES = [ROOT + "Final_validation/GSE98793_matrix_Transpose_labeled.csv",
            ROOT + "Final_validation/GSE63878_matrix_Transpose_labeled.csv"]
OUT = ROOT + "Enrichment_Reviewer12/"
LIBDIR, TAB, FIG = OUT + "libraries/", OUT + "tables/", OUT + "figures/"
for d in (LIBDIR, TAB, FIG): os.makedirs(d, exist_ok=True)

GENE_LISTS = {
    "Shared DEGs (19)": ["TDRD9", "OLFM4", "SYTL2", "PML", "BTN3A2", "PLXNA2", "EIF4B", "CYP2U1", "IGF1R", "GALNT11",
                         "HGF", "DKKL1", "GFI1B", "PTMA", "AHI1", "SCAPER", "SLC22A14", "DOCK4", "STAT2"],
    "Hub genes (10)": ["IGF1R", "STAT2", "PML", "HGF", "HDAC1", "JAK2", "JAK1", "MDM2", "ERBB2", "STAT3"]}
LIBRARIES = ["GO_Biological_Process_2025", "GO_Molecular_Function_2025", "GO_Cellular_Component_2025",
             "KEGG_2021_Human", "WikiPathways_2024_Human", "Reactome_Pathways_2024"]
MIN_SIZE, MAX_SIZE, FDR_CUT = 5, 2000, 0.05

# ---------------- gene universe: symbols measured on both platforms
def symbols(path):
    cols = pd.read_csv(path, nrows=0).columns
    s = {re.sub(r"\.\d+$", "", c).strip() for c in cols if c != "target"}
    return {g for g in s if g and "///" not in g}
UNIVERSE = set.intersection(*[symbols(p) for p in MATRICES])
N = len(UNIVERSE)
print(f"Gene universe: {N:,} genes measured on both platforms")

# ---------------- libraries (downloaded once, cached for reproducibility)
def get_lib(name):
    f = LIBDIR + name + ".gmt"
    if not os.path.exists(f):
        url = f"https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName={name}"
        r = requests.get(url, timeout=120); r.raise_for_status()
        open(f, "w").write(r.text)
    lib = {}
    for line in open(f):
        parts = line.rstrip("\n").split("\t")
        if len(parts) > 2:
            lib[parts[0]] = {g.split(",")[0].strip() for g in parts[2:] if g.strip()}
    return lib

def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = p[o] * n / np.arange(1, n + 1); q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n); out[o] = np.minimum(q, 1); return out

def enrich(genes, lib):
    G = set(genes) & UNIVERSE; n = len(G); rows = []
    for term, members in lib.items():
        T = members & UNIVERSE; K = len(T)
        if K < MIN_SIZE or K > MAX_SIZE: continue
        hit = sorted(G & T); k = len(hit)
        if k == 0: continue
        p = hypergeom.sf(k - 1, N, K, n)
        a, b, c, d = k, n - k, K - k, N - K - n + k
        orr = (a * d) / (b * c) if b * c > 0 else ((a + .5) * (d + .5)) / ((b + .5) * (c + .5))
        rows.append(dict(Term=term, Overlap=f"{k}/{K}", k=k, Term_size=K, Input_size=n, P_value=p,
                         Enrichment_ratio=(k / n) / (K / N), Odds_ratio=orr, Genes=";".join(hit)))
    df = pd.DataFrame(rows)
    if len(df):
        df["FDR"] = bh(df.P_value); df["Significant_FDR_0.05"] = df.FDR < FDR_CUT
        df = df.sort_values(["FDR", "P_value"]).reset_index(drop=True)
    return df

# ---------------- run
allres, summary = [], []
for lname in LIBRARIES:
    lib = get_lib(lname)
    for gl, genes in GENE_LISTS.items():
        df = enrich(genes, lib)
        df.insert(0, "Library", lname); df.insert(0, "Gene_list", gl)
        allres.append(df)
        summary.append(dict(Gene_list=gl, Library=lname, Terms_in_library=len(lib), Terms_tested=len(df),
                            Significant_FDR_0_05=int(df["Significant_FDR_0.05"].sum()) if len(df) else 0,
                            Min_FDR=round(df.FDR.min(), 4) if len(df) else np.nan))
res = pd.concat(allres, ignore_index=True)
summ = pd.DataFrame(summary)
missing = {gl: sorted(set(g) - UNIVERSE) for gl, g in GENE_LISTS.items()}

# ---------------- tables
cols = ["Gene_list", "Library", "Term", "Overlap", "P_value", "FDR", "Enrichment_ratio", "Odds_ratio", "Genes"]
res.to_csv(TAB + "S_enrichment_all_terms.csv", index=False, encoding="utf-8-sig")
sig = res[res["Significant_FDR_0.05"]][cols].copy()
for c in ["P_value", "FDR"]: sig[c] = sig[c].map(lambda v: f"{v:.2e}")
for c in ["Enrichment_ratio", "Odds_ratio"]: sig[c] = sig[c].round(1)
sig.to_csv(TAB + "Table_enrichment_significant_FDR0.05.csv", index=False, encoding="utf-8-sig")
summ.to_csv(TAB + "S_enrichment_summary.csv", index=False, encoding="utf-8-sig")
methods = pd.DataFrame([
    ("Gene universe", f"{N:,} genes measured on both platforms (GSE98793, GSE63878)"),
    ("Gene lists", "; ".join(f"{k}: {', '.join(v)}" for k, v in GENE_LISTS.items())),
    ("Genes not in universe", "; ".join(f"{k}: {', '.join(v) or 'none'}" for k, v in missing.items())),
    ("Libraries (Enrichr)", ", ".join(LIBRARIES)),
    ("Library download date", datetime.date.today().isoformat()),
    ("Test", "One-sided hypergeometric test; terms with at least one input gene"),
    ("Term size filter", f"{MIN_SIZE}-{MAX_SIZE} genes within the universe"),
    ("Multiple testing", f"Benjamini-Hochberg FDR within each library and gene list; significance FDR < {FDR_CUT}"),
    ("Enrichment ratio", "(k/n)/(K/N): k overlap, n input genes, K term size, N universe size")],
    columns=["Item", "Setting"])
methods.to_csv(TAB + "S_enrichment_methods.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter(TAB + "Enrichment_all_tables.xlsx") as xw:
    sig.to_excel(xw, sheet_name="Significant_FDR0.05", index=False)
    summ.to_excel(xw, sheet_name="Summary", index=False)
    methods.to_excel(xw, sheet_name="Methods", index=False)
    for gl in GENE_LISTS:
        res[res.Gene_list == gl].to_excel(xw, sheet_name=("All_" + gl.split(" (")[0].replace(" ", "_"))[:31], index=False)

# ---------------- figures: top terms per gene list, significance shown explicitly
short = {"GO_Biological_Process_2025": "GO BP", "GO_Molecular_Function_2025": "GO MF", "GO_Cellular_Component_2025": "GO CC",
         "KEGG_2021_Human": "KEGG", "WikiPathways_2024_Human": "WikiPathways", "Reactome_Pathways_2024": "Reactome"}
plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False, "font.family": "DejaVu Sans"})
for gl in GENE_LISTS:
    d = res[res.Gene_list == gl].sort_values("FDR").head(20).iloc[::-1]
    if not len(d): continue
    fig, ax = plt.subplots(figsize=(7.5, 0.3 * len(d) + 1.4))
    colors = ["#C0392B" if s else "#B3B6B7" for s in d["Significant_FDR_0.05"]]
    ax.barh(range(len(d)), -np.log10(d.FDR), color=colors, edgecolor="black", linewidth=0.3)
    ax.axvline(-np.log10(FDR_CUT), ls="--", color="#1F4E99", lw=0.8)
    clean = [re.sub(r" \(GO:\d+\)", "", t)[:60] for t in d.Term]
    lab = [f"{c} [{short[l]}; {o}]" for c, l, o in zip(clean, d.Library, d.Overlap)]
    ax.set_yticks(range(len(d))); ax.set_yticklabels(lab, fontsize=7)
    ax.set_xlabel("\u2212log10(FDR)   (dashed line: FDR = 0.05; red: significant, grey: not significant)")
    ax.set_title(f"Top enriched terms: {gl}", fontweight="bold")
    fig.tight_layout()
    nm = "Fig5_enrichment_" + gl.split(" (")[0].replace(" ", "_")
    for ext in ("png", "tiff"): fig.savefig(FIG + nm + "." + ext, dpi=600, bbox_inches="tight", **({"pil_kwargs": {"compression": "tiff_lzw"}} if ext == "tiff" else {}))
    fig.savefig(FIG + nm + ".pdf", bbox_inches="tight"); plt.close(fig)

pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 70)
print("\nGenes not in universe:", missing)
print("\nSUMMARY\n", summ.to_string(index=False))
for gl in GENE_LISTS:
    print(f"\nTOP 10 TERMS - {gl}\n", res[res.Gene_list == gl].sort_values("FDR").head(10)[["Library", "Term", "Overlap", "FDR", "Enrichment_ratio", "Genes"]].to_string(index=False))
print("\nSaved to", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Gene universe: 16,911 genes measured on both platforms

Genes not in universe: {'Shared DEGs (19)': [], 'Hub genes (10)': []}

SUMMARY
        Gene_list                    Library  Terms_in_library  Terms_tested  Significant_FDR_0_05  Min_FDR
Shared DEGs (19) GO_Biological_Process_2025              5343           220                     0   0.1048
  Hub genes (10) GO_Biological_Process_2025              5343           440                   335   0.0000
Shared DEGs (19) GO_Molecular_Function_2025              1174            45                     0   0.1001
  Hub genes (10) GO_Molecular_Function_2025              1174            51                    41   0.0000
Shared DEGs (19) GO_Cellular_Component_2025               468            30                     0   0.1235
  Hub genes (10) GO_Cellular_Component_2025               468            24                  

In [ ]:
# =====================================================================================
# ENRICHMENT ANALYSIS FOR REVIEWER #2, CONCERN #12  (Colab, plain Python)
#   * Gene universe: genes measured on both platforms (GSE98793 and GSE63878)
#   * Main libraries: Enrichr GO 2025 (BP/MF/CC), KEGG 2021, WikiPathways 2024
#   * Supplementary libraries: Reactome 2024, BioCarta 2016, BioPlanet 2019, DSigDB
#   * Method: one-sided hypergeometric test, Benjamini-Hochberg FDR within each library
#   * Reports: overlap, term size, p, FDR, enrichment ratio, odds ratio, contributing genes
#   * Two gene lists analysed separately: 19 shared DEGs and 10 hub genes
# =====================================================================================
import os, re, datetime, numpy as np, pandas as pd, requests, matplotlib.pyplot as plt
from scipy.stats import hypergeom
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/research/p5-stress_depression/p5_stress_depression_journal-20260927T103230Z-1-001/p5_stress_depression_journal/"
MATRICES = [ROOT + "Final_validation/GSE98793_matrix_Transpose_labeled.csv",
            ROOT + "Final_validation/GSE63878_matrix_Transpose_labeled.csv"]
OUT = ROOT + "Enrichment_Reviewer12/"
LIBDIR, TAB, FIG = OUT + "libraries/", OUT + "tables/", OUT + "figures/"
for d in (LIBDIR, TAB, FIG): os.makedirs(d, exist_ok=True)

GENE_LISTS = {
    "Shared DEGs (19)": ["TDRD9", "OLFM4", "SYTL2", "PML", "BTN3A2", "PLXNA2", "EIF4B", "CYP2U1", "IGF1R", "GALNT11",
                         "HGF", "DKKL1", "GFI1B", "PTMA", "AHI1", "SCAPER", "SLC22A14", "DOCK4", "STAT2"],
    "Hub genes (10)": ["IGF1R", "STAT2", "PML", "HGF", "HDAC1", "JAK2", "JAK1", "MDM2", "ERBB2", "STAT3"]}
LIBRARIES = ["GO_Biological_Process_2025", "GO_Molecular_Function_2025", "GO_Cellular_Component_2025",
             "KEGG_2021_Human", "WikiPathways_2024_Human",
             "Reactome_Pathways_2024", "BioCarta_2016", "BioPlanet_2019", "DSigDB"]
MAIN_LIBS = LIBRARIES[:5]            # main Fig 5: GO, KEGG, WikiPathways; the rest -> supplementary
MIN_SIZE, MAX_SIZE, FDR_CUT = 5, 2000, 0.05

# ---------------- gene universe: symbols measured on both platforms
def symbols(path):
    cols = pd.read_csv(path, nrows=0).columns
    s = {re.sub(r"\.\d+$", "", c).strip() for c in cols if c != "target"}
    return {g for g in s if g and "///" not in g}
UNIVERSE = set.intersection(*[symbols(p) for p in MATRICES])
N = len(UNIVERSE)
print(f"Gene universe: {N:,} genes measured on both platforms")

# ---------------- libraries (downloaded once, cached for reproducibility)
def get_lib(name):
    f = LIBDIR + name + ".gmt"
    if not os.path.exists(f):
        url = f"https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName={name}"
        r = requests.get(url, timeout=120); r.raise_for_status()
        open(f, "w").write(r.text)
    lib = {}
    for line in open(f):
        parts = line.rstrip("\n").split("\t")
        if len(parts) > 2:
            lib[parts[0]] = {g.split(",")[0].strip() for g in parts[2:] if g.strip()}
    return lib

def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = p[o] * n / np.arange(1, n + 1); q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n); out[o] = np.minimum(q, 1); return out

def enrich(genes, lib):
    G = set(genes) & UNIVERSE; n = len(G); rows = []
    for term, members in lib.items():
        T = members & UNIVERSE; K = len(T)
        if K < MIN_SIZE or K > MAX_SIZE: continue
        hit = sorted(G & T); k = len(hit)
        if k == 0: continue
        p = hypergeom.sf(k - 1, N, K, n)
        a, b, c, d = k, n - k, K - k, N - K - n + k
        orr = (a * d) / (b * c) if b * c > 0 else ((a + .5) * (d + .5)) / ((b + .5) * (c + .5))
        rows.append(dict(Term=term, Overlap=f"{k}/{K}", k=k, Term_size=K, Input_size=n, P_value=p,
                         Enrichment_ratio=(k / n) / (K / N), Odds_ratio=orr, Genes=";".join(hit)))
    df = pd.DataFrame(rows)
    if len(df):
        df["FDR"] = bh(df.P_value); df["Significant_FDR_0.05"] = df.FDR < FDR_CUT
        df = df.sort_values(["FDR", "P_value"]).reset_index(drop=True)
    return df

# ---------------- run
allres, summary = [], []
for lname in LIBRARIES:
    lib = get_lib(lname)
    for gl, genes in GENE_LISTS.items():
        df = enrich(genes, lib)
        df.insert(0, "Library", lname); df.insert(0, "Gene_list", gl)
        allres.append(df)
        summary.append(dict(Gene_list=gl, Library=lname, Terms_in_library=len(lib), Terms_tested=len(df),
                            Significant_FDR_0_05=int(df["Significant_FDR_0.05"].sum()) if len(df) else 0,
                            Min_FDR=round(df.FDR.min(), 4) if len(df) else np.nan))
res = pd.concat(allres, ignore_index=True)
summ = pd.DataFrame(summary)
missing = {gl: sorted(set(g) - UNIVERSE) for gl, g in GENE_LISTS.items()}

# ---------------- tables
cols = ["Gene_list", "Library", "Term", "Overlap", "P_value", "FDR", "Enrichment_ratio", "Odds_ratio", "Genes"]
res.to_csv(TAB + "S_enrichment_all_terms.csv", index=False, encoding="utf-8-sig")
sig = res[res["Significant_FDR_0.05"]][cols].copy()
for c in ["P_value", "FDR"]: sig[c] = sig[c].map(lambda v: f"{v:.2e}")
for c in ["Enrichment_ratio", "Odds_ratio"]: sig[c] = sig[c].round(1)
sig.to_csv(TAB + "Table_enrichment_significant_FDR0.05.csv", index=False, encoding="utf-8-sig")
summ.to_csv(TAB + "S_enrichment_summary.csv", index=False, encoding="utf-8-sig")
methods = pd.DataFrame([
    ("Gene universe", f"{N:,} genes measured on both platforms (GSE98793, GSE63878)"),
    ("Gene lists", "; ".join(f"{k}: {', '.join(v)}" for k, v in GENE_LISTS.items())),
    ("Genes not in universe", "; ".join(f"{k}: {', '.join(v) or 'none'}" for k, v in missing.items())),
    ("Main libraries (Enrichr)", ", ".join(MAIN_LIBS)),
    ("Supplementary libraries (Enrichr)", ", ".join(LIBRARIES[5:])),
    ("Library download date", datetime.date.today().isoformat()),
    ("Test", "One-sided hypergeometric test; terms with at least one input gene"),
    ("Term size filter", f"{MIN_SIZE}-{MAX_SIZE} genes within the universe"),
    ("Multiple testing", f"Benjamini-Hochberg FDR within each library and gene list; significance FDR < {FDR_CUT}"),
    ("Enrichment ratio", "(k/n)/(K/N): k overlap, n input genes, K term size, N universe size")],
    columns=["Item", "Setting"])
methods.to_csv(TAB + "S_enrichment_methods.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter(TAB + "Enrichment_all_tables.xlsx") as xw:
    sig.to_excel(xw, sheet_name="Significant_FDR0.05", index=False)
    summ.to_excel(xw, sheet_name="Summary", index=False)
    methods.to_excel(xw, sheet_name="Methods", index=False)
    for gl in GENE_LISTS:
        res[res.Gene_list == gl].to_excel(xw, sheet_name=("All_" + gl.split(" (")[0].replace(" ", "_"))[:31], index=False)

# ---------------- figures: main (GO/KEGG/WikiPathways) and supplementary (other libraries)
short = {"GO_Biological_Process_2025": "GO BP", "GO_Molecular_Function_2025": "GO MF", "GO_Cellular_Component_2025": "GO CC",
         "KEGG_2021_Human": "KEGG", "WikiPathways_2024_Human": "WikiPathways", "Reactome_Pathways_2024": "Reactome",
         "BioCarta_2016": "BioCarta", "BioPlanet_2019": "BioPlanet", "DSigDB": "DSigDB"}
plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False, "font.family": "DejaVu Sans"})
def bar(d, title, nm):
    d = d.sort_values("FDR").head(20).iloc[::-1]
    if not len(d): return
    fig, ax = plt.subplots(figsize=(7.5, 0.3 * len(d) + 1.4))
    ax.barh(range(len(d)), -np.log10(d.FDR), color=["#C0392B" if s else "#B3B6B7" for s in d["Significant_FDR_0.05"]],
            edgecolor="black", linewidth=0.3)
    ax.axvline(-np.log10(FDR_CUT), ls="--", color="#1F4E99", lw=0.8)
    clean = [re.sub(r" \(GO:\d+\)", "", t)[:60] for t in d.Term]
    ax.set_yticks(range(len(d))); ax.set_yticklabels([f"{c} [{short[l]}; {o}]" for c, l, o in zip(clean, d.Library, d.Overlap)], fontsize=7)
    ax.set_xlabel("\u2212log10(FDR)   (dashed line: FDR = 0.05; red: significant, grey: not significant)")
    ax.set_title(title, fontweight="bold"); fig.tight_layout()
    for ext in ("png", "tiff"): fig.savefig(FIG + nm + "." + ext, dpi=600, bbox_inches="tight", **({"pil_kwargs": {"compression": "tiff_lzw"}} if ext == "tiff" else {}))
    fig.savefig(FIG + nm + ".pdf", bbox_inches="tight"); plt.close(fig)

for gl in GENE_LISTS:
    tag = gl.split(" (")[0].replace(" ", "_")
    r = res[res.Gene_list == gl]
    bar(r[r.Library.isin(MAIN_LIBS)], f"GO, KEGG and WikiPathways: {gl}", f"Fig5_enrichment_{tag}")
    for lib in LIBRARIES[5:]:
        bar(r[r.Library == lib], f"{short[lib]}: {gl}", f"FigS_enrichment_{short[lib]}_{tag}")

pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 70)
print("\nGenes not in universe:", missing)
print("\nSUMMARY\n", summ.to_string(index=False))
for gl in GENE_LISTS:
    print(f"\nTOP 10 TERMS - {gl}\n", res[res.Gene_list == gl].sort_values("FDR").head(10)[["Library", "Term", "Overlap", "FDR", "Enrichment_ratio", "Genes"]].to_string(index=False))
print("\nSaved to", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Gene universe: 16,911 genes measured on both platforms

Genes not in universe: {'Shared DEGs (19)': [], 'Hub genes (10)': []}

SUMMARY
        Gene_list                    Library  Terms_in_library  Terms_tested  Significant_FDR_0_05  Min_FDR
Shared DEGs (19) GO_Biological_Process_2025              5343           220                     0   0.1048
  Hub genes (10) GO_Biological_Process_2025              5343           440                   335   0.0000
Shared DEGs (19) GO_Molecular_Function_2025              1174            45                     0   0.1001
  Hub genes (10) GO_Molecular_Function_2025              1174            51                    41   0.0000
Shared DEGs (19) GO_Cellular_Component_2025               468            30                     0   0.1235
  Hub genes (10) GO_Cellular_Component_2025               468            24                    10   0.0061
Shared DEGs (19)            KEGG_2021_Human               320            54                     4   0.0335
  Hub ge

In [ ]:
import pandas as pd
TAB = "/content/drive/MyDrive/research/p5-stress_depression/p5_stress_depression_journal-20260927T103230Z-1-001/p5_stress_depression_journal/Enrichment_Reviewer12/tables/"
res = pd.read_csv(TAB + "S_enrichment_all_terms.csv")

cols = ["Gene_list", "Library", "Term", "Overlap", "P_value", "FDR", "Significant_FDR_0.05", "Enrichment_ratio", "Odds_ratio", "Genes"]
fmt = lambda d: d.assign(P_value=d.P_value.map(lambda v: f"{v:.2e}"), FDR=d.FDR.map(lambda v: f"{v:.2e}"),
                         Enrichment_ratio=d.Enrichment_ratio.round(1), Odds_ratio=d.Odds_ratio.round(1))

# S15: top 10 terms per gene list and library (readable)
top10 = (res.sort_values("FDR").groupby(["Gene_list", "Library"], sort=False).head(10)
            .sort_values(["Gene_list", "Library", "FDR"]))[cols]
with pd.ExcelWriter(TAB + "S15_Table_enrichment_top10.xlsx") as xw:
    for gl, d in top10.groupby("Gene_list", sort=False):
        fmt(d).to_excel(xw, sheet_name=gl.split(" (")[0][:31], index=False)

# S16: all tested terms (one sheet per gene list x library) + settings + summary
with pd.ExcelWriter(TAB + "S16_File_enrichment_all_terms.xlsx") as xw:
    pd.read_csv(TAB + "S_enrichment_methods.csv").to_excel(xw, sheet_name="Settings", index=False)
    pd.read_csv(TAB + "S_enrichment_summary.csv").to_excel(xw, sheet_name="Summary", index=False)
    for (gl, lib), d in res.groupby(["Gene_list", "Library"], sort=False):
        name = (("Shared_" if gl.startswith("Shared") else "Hub_") + lib.replace("_Human", ""))[:31]
        fmt(d.sort_values("FDR"))[cols].to_excel(xw, sheet_name=name, index=False)

print("S15 rows:", len(top10), "| S16 rows:", len(res))

S15 rows: 178 | S16 rows: 3543
